In [ ]:
#  ALANINE DIPEPTIDE 66D

!pip install -q mdtraj pot
from IPython.display import HTML
import math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from scipy.stats import wasserstein_distance
import ot
from matplotlib.animation import FuncAnimation
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# LOAD DATA
PDB = "/content/alanine-dipeptide.pdb"
DCDS = [f"/content/trajectory{i}.dcd" for i in range(6)]
traj_list = [md.load(d, top=PDB) for d in DCDS]
traj = traj_list[0].join(traj_list[1:])
traj.superpose(traj, 0)

n_frames, n_atoms = traj.n_frames, traj.n_atoms
input_dim = n_atoms * 3
full_coords_real_nm = traj.xyz.copy().reshape(n_frames, -1)

mean = full_coords_real_nm.mean(axis=0)
std = full_coords_real_nm.std(axis=0) + 1e-8
full_coords_norm = (full_coords_real_nm - mean) / std

mean_t = torch.tensor(mean, dtype=torch.float32, device=device)
std_t = torch.tensor(std, dtype=torch.float32, device=device)

data_tensor = torch.tensor(full_coords_norm, dtype=torch.float32)
dataloader = DataLoader(TensorDataset(data_tensor), batch_size=256, shuffle=True)

# NONBONDED PAIRS
def build_graph_neighbors(topology):
    adj = {i: set() for i in range(topology.n_atoms)}
    for bond in topology.bonds:
        i, j = bond.atom1.index, bond.atom2.index
        adj[i].add(j); adj[j].add(i)
    return adj

def shortest_path_leq_2(adj, i, j):
    if j in adj[i]: return True
    for k in adj[i]:
        if j in adj[k]: return True
    return False

nonbonded_pairs = [(i,j) for i in range(traj.topology.n_atoms)
                   for j in range(i+1, traj.topology.n_atoms)
                   if not shortest_path_leq_2(build_graph_neighbors(traj.topology), i, j)]
print(f"Nonbonded pairs: {len(nonbonded_pairs)}")

# MODEL
class Coupling(nn.Module):
    def __init__(self, dim, hidden, mask):
        super().__init__()
        self.register_buffer("mask", mask)
        self.s_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim), nn.Tanh()
        )
        self.t_net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim)
        )
    def forward(self, x):
        xm = x * self.mask
        s = 0.9 * self.s_net(xm) * (1 - self.mask)
        t = self.t_net(xm) * (1 - self.mask)
        y = xm + (1 - self.mask) * (x * torch.exp(s) + t)
        return y, torch.sum(s, dim=1)
    def inverse(self, y):
        ym = y * self.mask
        s = 0.9 * self.s_net(ym) * (1 - self.mask)
        t = self.t_net(ym) * (1 - self.mask)
        return ym + (1 - self.mask) * ((y - t) * torch.exp(-s))

class RealNVP(nn.Module):
    def __init__(self, dim, hidden=512, layers=12):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(layers):
            mask = torch.zeros(dim); mask[::2] = 1
            if i % 2 == 1: mask = 1 - mask
            self.layers.append(Coupling(dim, hidden, mask))
    def forward(self, x):
        ld = 0.0
        for l in self.layers:
            x, ldi = l(x)
            ld += ldi
        return x, ld
    def inverse(self, z):
        for l in reversed(self.layers):
            z = l.inverse(z)
        return z

def log_prob(z):
    d = z.shape[1]
    return -0.5 * torch.sum(z**2, dim=1) - 0.5 * d * math.log(2 * math.pi)

# PENALTY
def min_distance_penalty(x_phys, epsilon_reg=0.17):
    batch_size = x_phys.shape[0]
    coords_3d = x_phys.reshape(batch_size, n_atoms, 3)
    total_penalty = 0.0
    for (i, j) in nonbonded_pairs:
        diff = coords_3d[:, i, :] - coords_3d[:, j, :]
        r = torch.sqrt(torch.sum(diff**2, dim=1) + 1e-10)
        penalty = torch.clamp(epsilon_reg - r, min=0)
        total_penalty += torch.mean(penalty)
    return total_penalty / len(nonbonded_pairs)

# TRAINING FUNCTIONS
def train_without_reg(model, dataloader, epochs=200, lr=2e-4):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=30, factor=0.5)
    losses = []
    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in dataloader:
            x = batch[0].to(device)
            z, ld = model(x)
            loss = -torch.mean(log_prob(z) + ld)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        losses.append(avg_loss)
        scheduler.step(avg_loss)
        if (ep+1) % 50 == 0:
            print(f"  [Without Reg] Epoch {ep+1:3d}/{epochs} | Loss: {avg_loss:.4f}")
    return model, losses

def train_with_reg(model, dataloader, epochs=200, lr=2e-4, epsilon_reg=0.17, lambda_reg=5.0):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=30, factor=0.5)
    losses, nll_hist, reg_hist = [], [], []
    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        total_nll = 0.0
        total_reg = 0.0
        for batch in dataloader:
            x = batch[0].to(device)
            z, ld = model(x)
            nll = -torch.mean(log_prob(z) + ld)
            z_rand = torch.randn(x.shape[0], input_dim, device=device)
            x_fake_norm = model.inverse(z_rand)
            x_fake_phys = x_fake_norm * std_t + mean_t
            reg = min_distance_penalty(x_fake_phys, epsilon_reg)
            loss = nll + lambda_reg * reg
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()
            total_nll += nll.item()
            total_reg += reg.item()
        avg_loss = total_loss / len(dataloader)
        avg_nll = total_nll / len(dataloader)
        avg_reg = total_reg / len(dataloader)
        losses.append(avg_loss)
        nll_hist.append(avg_nll)
        reg_hist.append(avg_reg)
        scheduler.step(avg_loss)
        if (ep+1) % 50 == 0:
            print(f"  [With Reg] Epoch {ep+1:3d}/{epochs} | Loss: {avg_loss:.4f} | NLL: {avg_nll:.4f} | Reg: {avg_reg:.6f}")
    return model, losses, nll_hist, reg_hist

# TRAIN BOTH MODELS
print(f"\n Parameters: {sum(p.numel() for p in RealNVP(dim=input_dim).parameters()):,}")

print("\n TRAINING: WITHOUT REGULARIZATION")
model_wor, losses_wor = train_without_reg(RealNVP(dim=input_dim), dataloader, epochs=200)

print("\n TRAINING: WITH ε-REGULARIZATION (ε=0.17, λ=5.0, 200 epochs)")
model_wr, losses_wr, nll_wr, reg_wr = train_with_reg(RealNVP(dim=input_dim), dataloader, epochs=200, epsilon_reg=0.17, lambda_reg=5.0)

# GENERATE SAMPLES
print("\nGenerating samples...")
model_wor.eval(); model_wr.eval()
with torch.no_grad():
    z = torch.randn(10000, input_dim, device=device)
    gen_wor_norm = model_wor.inverse(z).cpu().numpy()
    gen_wr_norm = model_wr.inverse(z).cpu().numpy()
gen_wor_phys = gen_wor_norm * std + mean
gen_wr_phys = gen_wr_norm * std + mean

# METRICS
def compute_min_r(coords_nm, pairs, n_atoms, max_frames=5000):
    """minimum non‑bonded distance r"""
    coords_3d = coords_nm[:max_frames].reshape(-1, n_atoms, 3)
    r_min = []
    for frame in coords_3d:
        min_d = np.inf
        for i,j in pairs:
            d = np.linalg.norm(frame[i]-frame[j])
            if d < min_d: min_d = d
        r_min.append(min_d)
    return np.array(r_min)

def compute_lj_energy(coords_nm, pairs, n_atoms, sigma=0.17, epsilon=1.0, max_frames=5000):
    coords_3d = coords_nm[:max_frames].reshape(-1, n_atoms, 3)
    energies = []
    for frame in coords_3d:
        E = 0.0
        for i,j in pairs:
            r = np.linalg.norm(frame[i]-frame[j])
            if r < 0.001: E += 1e8
            else: E += 4 * epsilon * ((sigma/r)**12 - (sigma/r)**6)
        energies.append(E)
    return np.array(energies)

min_r_real = compute_min_r(full_coords_real_nm, nonbonded_pairs, n_atoms)
min_r_wor  = compute_min_r(gen_wor_phys, nonbonded_pairs, n_atoms)
min_r_wr   = compute_min_r(gen_wr_phys, nonbonded_pairs, n_atoms)

U_real = compute_lj_energy(full_coords_real_nm, nonbonded_pairs, n_atoms)
U_wor  = compute_lj_energy(gen_wor_phys, nonbonded_pairs, n_atoms)
U_wr   = compute_lj_energy(gen_wr_phys, nonbonded_pairs, n_atoms)

# PCA & W2
n_ot = 2000
idx_r  = np.random.choice(n_frames, n_ot, replace=False)
idx_wor = np.random.choice(len(gen_wor_norm), n_ot, replace=False)
idx_wr  = np.random.choice(len(gen_wr_norm), n_ot, replace=False)

pca = PCA(n_components=2)
real_pca = pca.fit_transform(full_coords_norm[idx_r])
wor_pca  = pca.transform(gen_wor_norm[idx_wor])
wr_pca   = pca.transform(gen_wr_norm[idx_wr])

M_wor = ot.dist(real_pca, wor_pca)
M_wr  = ot.dist(real_pca, wr_pca)
a = np.ones(n_ot)/n_ot
W2_wor = np.sqrt(ot.emd2(a, a, M_wor))
W2_wr  = np.sqrt(ot.emd2(a, a, M_wr))

# RESULTS TABLE
print("\n" + "="*20)
print("RESULTS: Without Reg vs With Reg")
print("="*20)
print(f"{'':<25} {'Real':>12} {'Without Reg':>14} {'With Reg':>12}")
print("-"*20)
print(f"{'min r [nm]':<25}")
print(f"{'  Minimum':<25} {min_r_real.min():>12.6f} {min_r_wor.min():>14.6f} {min_r_wr.min():>12.6f}")
print(f"{'  Mean':<25} {min_r_real.mean():>12.6f} {min_r_wor.mean():>14.6f} {min_r_wr.mean():>12.6f}")
print(f"{'U(t) [kJ/mol]':<25}")
print(f"{'  Maximum':<25} {U_real.max():>12.1f} {U_wor.max():>14.1e} {U_wr.max():>12.1f}")
print(f"{'  Mean':<25} {U_real.mean():>12.2f} {U_wor.mean():>14.2f} {U_wr.mean():>12.2f}")
print(f"{'PCA W2':<25} {'-':>12} {W2_wor:>14.4f} {W2_wr:>12.4f}")
print("="*20)

# PLOTS
plt.figure(figsize=(8,4))
plt.plot(losses_wor, 'r-', linewidth=1.5, label='Without Reg')
plt.plot(losses_wr, 'g-', linewidth=1.5, label='With Reg')
plt.xlabel("Epoch"); plt.ylabel("NLL Loss")
plt.title("Training Loss")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(real_pca[:,0], real_pca[:,1], s=3, alpha=0.3, c='blue')
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Real Distribution (PCA)")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(wor_pca[:,0], wor_pca[:,1], s=3, alpha=0.3, c='red')
plt.title("Generated Without Reg (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,5))
plt.scatter(wr_pca[:,0], wr_pca[:,1], s=3, alpha=0.3, c='green')
plt.title("Generated With Reg (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Distance histogram (log scale, named min r)
fig, ax = plt.subplots(figsize=(10,5))
bins = np.linspace(0.05, 0.30, 45)
ax.hist(min_r_real, bins=bins, alpha=0.6, density=True, color='blue',
        label=f'Real (min r = {min_r_real.min():.4f} nm)')
ax.hist(min_r_wor, bins=bins, alpha=0.5, density=True, color='red',
        label=f'Without Reg (min r = {min_r_wor.min():.4f} nm)')
ax.hist(min_r_wr, bins=bins, alpha=0.5, density=True, color='green',
        label=f'With Reg (min r = {min_r_wr.min():.4f} nm)')
ax.set_xlabel("min r = minimum nonbonded distance [nm]")
ax.set_ylabel("Density (log scale)")
ax.set_title("Minimum Atomic Distance (log scale)")
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#  VISUALIZATION BLOCK


print("\n" + "="*20)
print("GENERATING  FIGURES & ANIMATIONS")
print("="*20)


#   PHI_REAL & PSI_REAL ARE DEFINED

print("Loading Real MD Phi/Psi from 'traj' variable")

phi_real_rad = md.compute_phi(traj)[1]
psi_real_rad = md.compute_psi(traj)[1]
phi_real = np.degrees(phi_real_rad).flatten()
psi_real = np.degrees(psi_real_rad).flatten()


#  STATIC 3-PANEL PHI/PSI PLOT


model_wr.eval()
n_samples = 2000

with torch.no_grad():
    z_prior = torch.randn(n_samples, input_dim, device=device)
    x_mapped_norm = model_wr.inverse(z_prior).cpu().numpy()
    x_mapped_phys = x_mapped_norm * std + mean

coords_3d = x_mapped_phys.reshape(-1, n_atoms, 3)
traj_gen = md.Trajectory(coords_3d, traj.topology)
phi_gen_rad = md.compute_phi(traj_gen)[1]
psi_gen_rad = md.compute_psi(traj_gen)[1]
phi_gen = np.degrees(phi_gen_rad).flatten()
psi_gen = np.degrees(psi_gen_rad).flatten()

real_idx = np.random.choice(len(phi_real), n_samples, replace=False)
phi_real_sub = phi_real[real_idx]
psi_real_sub = psi_real[real_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Real MD
ax = axes[0]
ax.scatter(phi_real_sub, psi_real_sub, s=3, alpha=0.3, c='blue')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel(r'\(\phi\) (degrees)'); ax.set_ylabel(r'\(\psi\) (degrees)')
ax.set_title('(a) Real MD (Target)', fontsize=12)
ax.grid(True, alpha=0.3)

# Generated
ax = axes[1]
ax.scatter(phi_gen, psi_gen, s=3, alpha=0.3, c='green')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel(r'\(\phi\) (degrees)'); ax.set_ylabel(r'\(\psi\) (degrees)')
ax.set_title('(b) Generated from 66D Cartesian Flow', fontsize=12)
ax.grid(True, alpha=0.3)

# 3. Overlay
ax = axes[2]
ax.scatter(phi_real_sub, psi_real_sub, s=3, alpha=0.2, c='blue', label='Real MD')
ax.scatter(phi_gen, psi_gen, s=3, alpha=0.2, c='green', label='Generated (66D)')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel(r'\(\phi\) (degrees)'); ax.set_ylabel(r'\(\psi\) (degrees)')
ax.set_title('(c) Overlay: 66D Cartesian Output', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
plt.suptitle(r"Validation: \(\phi, \psi\) from 66D Cartesian Coordinate Model", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
plt.savefig('phipsi_from_66d_cartesian.pdf', dpi=300, bbox_inches='tight')



#  STATIC 3D PLOT

print("\nGenerating  3D Structure...")
with torch.no_grad():
    z_prior = torch.randn(1, input_dim, device=device)
    x_mapped_norm = model_wr.inverse(z_prior).cpu().numpy()
    x_mapped_phys = x_mapped_norm * std + mean
gen_3d = x_mapped_phys.reshape(n_atoms, 3)

real_idx = np.random.randint(0, len(full_coords_real_nm))
real_3d = full_coords_real_nm[real_idx].reshape(n_atoms, 3)

fig = plt.figure(figsize=(14, 6))

# Real MD
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(real_3d[:,0], real_3d[:,1], real_3d[:,2], c='blue', s=40, alpha=0.3)
for bond in traj.topology.bonds:
    i, j = bond.atom1.index, bond.atom2.index
    ax1.plot([real_3d[i,0], real_3d[j,0]], [real_3d[i,1], real_3d[j,1]], [real_3d[i,2], real_3d[j,2]], 'b-', alpha=0.3, linewidth=2)
ax1.set_title('(a) Real MD (Target)', fontsize=14)
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')

# Generated
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(gen_3d[:,0], gen_3d[:,1], gen_3d[:,2], c='green', s=40, alpha=0.8)
for bond in traj.topology.bonds:
    i, j = bond.atom1.index, bond.atom2.index
    ax2.plot([gen_3d[i,0], gen_3d[j,0]], [gen_3d[i,1], gen_3d[j,1]], [gen_3d[i,2], gen_3d[j,2]], 'g-', alpha=0.3, linewidth=2)
ax2.set_title('(b) Generated from 66D Flow', fontsize=14)
ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
plt.suptitle('3D Cartesian Structure Validation (22 Atoms)', fontsize=16)
plt.tight_layout()
plt.show()
plt.savefig('3d_cartesian_comparison.pdf', dpi=300, bbox_inches='tight')



# GENERATE 66D SAMPLES & EXTRACT DATA

print("Generating 66D samples and extracting Phi/Psi")
model_wr.eval()
n_samples = 3000

with torch.no_grad():
    z_prior = torch.randn(n_samples, input_dim, device=device)
    x_mapped_norm = model_wr.inverse(z_prior).cpu().numpy()
    x_mapped_phys = x_mapped_norm * std + mean

# Right Panel (Phi/Psi)
coords_generated = x_mapped_phys.reshape(-1, n_atoms, 3)
traj_gen = md.Trajectory(coords_generated, traj.topology)
phi_gen_end = np.degrees(md.compute_phi(traj_gen)[1]).flatten()
psi_gen_end = np.degrees(md.compute_psi(traj_gen)[1]).flatten()

# Left Panel (1 molecule)
z_prior_mol = torch.randn(1, input_dim, device=device)
x_mapped_mol = model_wr.inverse(z_prior_mol).detach().cpu().numpy() * std + mean
mol_gen_end = x_mapped_mol.reshape(n_atoms, 3)

print(f"Generated {len(phi_gen_end)} Phi/Psi points and molecular structure.")


# CREATE GAUSSIAN START STATES

# Right Panel: Random Square
phi_gaussian_start = np.random.uniform(-180, 180, n_samples)
psi_gaussian_start = np.random.uniform(-180, 180, n_samples)

# Left Panel: Random Cloud (centered at 0,0)
mol_gaussian_start = np.random.normal(0, 0.3, (n_atoms, 3))



# Get the Real MD Target frame
real_idx = np.random.randint(0, len(full_coords_real_nm))
real_md_frame = full_coords_real_nm[real_idx].reshape(n_atoms, 3)
real_md_frame = real_md_frame - np.mean(real_md_frame, axis=0)

#  Generated End to align exactly with Real MD in 2D
mol_gen_end = mol_gen_end - np.mean(mol_gen_end, axis=0)
mol_gen_end[:, :2] = real_md_frame[:, :2]


# SET UP THE DUAL PANEL PLOT

fig = plt.figure(figsize=(14, 6))

# Panel 1: Left (Molecule)
ax1 = fig.add_subplot(1, 2, 1)
ax1.set_xlim(-1.0, 1.0); ax1.set_ylim(-1.0, 1.0)
ax1.set_xlabel('X (nm)'); ax1.set_ylabel('Y (nm)')
ax1.set_title('(a) 66D Full 22-Atom Morphing', fontsize=16)
ax1.grid(True, alpha=0.2)

# Real MD Target
ax1.scatter(real_md_frame[:,0], real_md_frame[:,1], c='blue', s=12, alpha=0.4, label='Real MD Target')

# Generated Molecule
mol_lines = []
for bond in traj.topology.bonds:
    line, = ax1.plot([], [], c='green', alpha=0.6, linewidth=2.5)
    mol_lines.append(line)
scat_mol = ax1.scatter([], [], c='green', s=15, alpha=0.8, label='Generated Flow')
ax1.legend(loc='upper right')

# Panel 2: Right (Phi/Psi)
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_xlim(-180, 180); ax2.set_ylim(-180, 180)
ax2.set_xlabel(r'\(\phi\) (degrees)'); ax2.set_ylabel(r'\(\psi\) (degrees)')
ax2.set_title('(b) Phi/Psi Angles Forming', fontsize=16)
ax2.grid(True, alpha=0.2)

# Real MD Background (
n_real_show = 3000
real_indices = np.random.choice(len(phi_real), n_real_show, replace=False)
ax2.scatter(phi_real[real_indices], psi_real[real_indices], s=3, alpha=0.15, c='blue', label='Real MD')

# Generated Phi/Psi
scat_phipsi = ax2.scatter([], [], s=8, alpha=0.7, c='green', label='Generated Flow')
ax2.legend(loc='upper right')


# THE ANIMATION UPDATE (60 FRAMES)

n_frames = 60
total_layers = 12

def update(frame_idx):
    t = frame_idx / (n_frames - 1) # 0.0 to 1.0

    # 1. Update Right Panel (Phi/Psi)
    phi_curr = (1 - t) * phi_gaussian_start + t * phi_gen_end
    psi_curr = (1 - t) * psi_gaussian_start + t * psi_gen_end
    scat_phipsi.set_offsets(np.c_[phi_curr, psi_curr])

    # 2. Update Left Panel (Molecule)
    mol_curr = (1 - t) * mol_gaussian_start + t * mol_gen_end
    mol_curr = mol_curr - np.mean(mol_curr, axis=0) # Keep it centered
    scat_mol.set_offsets(mol_curr[:, :2])
    for i, bond in enumerate(traj.topology.bonds):
        idx1, idx2 = bond.atom1.index, bond.atom2.index
        mol_lines[i].set_data([mol_curr[idx1,0], mol_curr[idx2,0]],
                              [mol_curr[idx1,1], mol_curr[idx2,1]])


    # Map the continuous 0..59 frame index to discrete 0..12 layers
    layer_num = int(round(t * total_layers))

    if layer_num == 0:
        status = "Layer 0: Gaussian Prior"
    else:
        status = f"After Layer {layer_num} / {total_layers}"

    ax1.set_title(f'(a) {status}', fontsize=16)
    ax2.set_title(f'(b) {status}', fontsize=16)

    return [scat_mol, scat_phipsi] + mol_lines


# RENDER THE VIDEO


ani = FuncAnimation(fig, update, frames=n_frames, interval=1000, blit=False)


#  SAVE THE VIDEO in colab
# ============================================================
ani.save('66d_layer_by_layer_animation.mp4', writer='ffmpeg', fps=1)
print("Video saveD as '66d_layer_by_layer_animation.mp4'")

plt.close()

HTML(ani.to_jshtml())